# Self-Play V10 (Colab)

Generates self-play training data for V10 using **feature-value MCTS**
(NN value head is broken on val_weight=0 models — see HISTORY lessons
112-114). Model: pillar2w2_epoch_10.

**Critical:** BATCH_SIZE=8 — anything larger destroys MCTS quality
via virtual-loss starvation (HISTORY lesson 115).

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `pillar2w2_epoch_10.pt` — model
3. `feature_value_weights.npz` — feature-value evaluator

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

MODEL_NAME = 'pillar2w2_epoch_10.pt'

os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/{MODEL_NAME} /content/alphatrain/data/model.pt
!cp {DRIVE}/feature_value_weights.npz /content/alphatrain/data/feature_value_weights.npz
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')
print(f'Feature weights: {os.path.getsize("/content/alphatrain/data/feature_value_weights.npz")} bytes')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from alphatrain.evaluate import load_model
net, ms = load_model('/content/alphatrain/data/model.pt', 'cpu')
del net
print(f'Model OK, max_score={ms}')

In [ ]:
# === SELF-PLAY V10 ===
# Distribute across instances by editing SEED_START / SEED_END:
# Instance 1: 800000-800500
# Instance 2: 800500-801000
# Instance 3: 801000-801500
# Instance 4: 801500-802000
SEED_START = 800000
SEED_END = 800500
# =====================

SIMS = 1600              # Quality target (~62% turn-cap on M5 Max eval)
WORKERS = 24
BATCH_SIZE = 8           # MCTS quality requires bs=8 (HISTORY lesson 115)
MAX_TURNS = 5000
SAVE_DIR = f'{DRIVE}/selfplay_v10_s1600'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Seeds: {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} games)')
print(f'Sims: {SIMS}, Cap: {MAX_TURNS} turns, Workers: {WORKERS}, bs: {BATCH_SIZE}')
print(f'Save: {SAVE_DIR}')

!python -m alphatrain.scripts.selfplay \
    --model /content/alphatrain/data/model.pt \
    --feature-value-weights /content/alphatrain/data/feature_value_weights.npz \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --sims {SIMS} --batch-size {BATCH_SIZE} \
    --save-dir {SAVE_DIR} \
    --workers {WORKERS} --device cuda \
    --temperature-moves 5 \
    --max-turns {MAX_TURNS} \
    --compile